# WER · CER 이해하기

음성 인식(STT) 결과가 **정답 문장과 얼마나 다른지**를 숫자로 나타내는 방법을 처음부터 직접 만들어 봅니다.

- 8강(WER·CER와 오류 분석)은 수업에서 다루지 않아서, 프로젝트(`scoring.py`)에 쓰기 위해 **자습으로 정리**한 노트북입니다.
- 이 노트북의 코드는 모두 직접 실행해서 결과를 확인했습니다. 확인하지 못한 내용은 **[추정]** 또는 **[일반 지식]** 으로 표시했습니다.
- 사용 패키지는 `pandas`와 표준 라이브러리(`re`, `unicodedata`, `difflib`, `random`)뿐이라 `voice-ai` 환경에서 바로 실행됩니다.

| 순서 | 내용 |
|---|---|
| 1 | WER·CER가 무엇을 재는가 (치환·삭제·삽입) |
| 2 | 편집 거리(Levenshtein) 표를 직접 채우기 |
| 3 | 역추적으로 "어디가 틀렸는지" 찾기 |
| 4 | WER·CER 함수 만들고 손계산과 맞춰 보기 |
| 5 | WER가 1을 넘는 경우 |
| 6 | 단어 단위 vs 글자 단위 (영어·한국어) |
| 7 | 정규화가 점수를 바꾼다 |
| 8 | `difflib`와 편집 거리는 다를 수 있다 |
| 9 | 연습 문제 |
| 10 | 정리와 프로젝트 연결 |

## 1. WER · CER가 무엇을 재는가

STT가 이렇게 인식했다고 해 봅시다.

| | 문장 |
|---|---|
| 정답(reference) | the cat sat on the mat |
| 인식(hypothesis) | the cat sit on mat |

사람이 보면 틀린 곳이 두 군데입니다.

- `sat` → `sit` : 다른 단어로 **바뀜** → **치환(Substitution, S)**
- `the` 하나가 **빠짐** → **삭제(Deletion, D)**
- (인식 쪽에 정답에 없는 단어가 **끼어들면** → **삽입(Insertion, I)**)

$$\text{WER} = \frac{S + D + I}{N}$$

- $N$ = **정답 문장의 단어 수**(인식 문장의 단어 수가 아님)
- 위 예: $S=1,\ D=1,\ I=0,\ N=6$ 이므로 WER $= 2/6 \approx 0.333$

**CER(Character Error Rate)** 은 같은 식을 **단어 대신 글자** 단위로 계산한 것입니다.

## 2. 편집 거리(Levenshtein distance) 표 채우기

"정답을 인식 결과로 바꾸려면 **치환·삭제·삽입을 최소 몇 번** 해야 하는가"가 편집 거리이고, 이 최소 횟수가 곧 $S+D+I$ 입니다.

표 `d[i][j]`의 뜻: **정답의 앞 `i`개와 인식의 앞 `j`개를 맞추는 데 필요한 최소 횟수**

- 첫 행·첫 열: 한쪽이 비어 있으면 상대 길이만큼 삭제(또는 삽입)해야 하므로 `d[i][0] = i`, `d[0][j] = j`
- 나머지 칸: 아래 세 값 중 **최솟값**
  - 대각선 `d[i-1][j-1]` + (두 토큰이 같으면 0, 다르면 1) → 일치/치환
  - 위쪽 `d[i-1][j]` + 1 → 정답 토큰을 지움(삭제)
  - 왼쪽 `d[i][j-1]` + 1 → 인식 토큰을 끼워 넣음(삽입)

In [1]:
import pandas as pd

def edit_matrix(ref, hyp):
    """ref, hyp: 토큰 리스트(단어 리스트 또는 글자 리스트). 편집 거리 표(2차원 리스트)를 돌려줌."""
    n, m = len(ref), len(hyp)
    d = [[0] * (m + 1) for _ in range(n + 1)]
    for i in range(n + 1):
        d[i][0] = i                      # 정답 i개를 모두 지워야 함
    for j in range(m + 1):
        d[0][j] = j                      # 인식 j개를 모두 끼워 넣어야 함
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            cost = 0 if ref[i - 1] == hyp[j - 1] else 1
            d[i][j] = min(d[i - 1][j] + 1,          # 삭제
                          d[i][j - 1] + 1,          # 삽입
                          d[i - 1][j - 1] + cost)   # 일치 또는 치환
    return d

def show_matrix(ref, hyp):
    d = edit_matrix(ref, hyp)
    return pd.DataFrame(d, index=["(시작)"] + ref, columns=["(시작)"] + hyp)

ref = "the cat sat on the mat".split()
hyp = "the cat sit on mat".split()
show_matrix(ref, hyp)

,(시작),the,cat,sit,on,mat
(시작),0,1,2,3,4,5
the,1,0,1,2,3,4
cat,2,1,0,1,2,3
sat,3,2,1,1,2,3
on,4,3,2,2,1,2
the,5,4,3,3,2,2
mat,6,5,4,4,3,2


**표 읽는 법**

- 오른쪽 아래 칸(`mat` 행, `mat` 열)의 값이 **전체 편집 거리**입니다. 위 표에서는 **2** 입니다.
- 대각선으로 내려가며 값이 그대로면 일치, 1 늘어나면 치환입니다. 위쪽에서 내려오며 1 늘면 삭제, 왼쪽에서 오며 1 늘면 삽입입니다.
- 이 표만으로는 **거리(몇 번 틀렸나)** 는 알지만 **어디가 어떻게 틀렸나**는 모릅니다. 다음 절에서 표를 거꾸로 따라가 알아냅니다.

## 3. 역추적(backtracking) — 어디가 틀렸는지 찾기

표의 오른쪽 아래에서 출발해 "이 값이 어느 칸에서 왔는가"를 거꾸로 따라가면 치환·삭제·삽입 목록을 얻습니다.

- 같은 거리를 만드는 경로가 **여러 개**일 수 있습니다. 어느 것을 고를지는 `if` 순서(여기서는 일치/치환 → 삭제 → 삽입)로 정해집니다. 거리(점수)는 같고, **오류의 위치 표시가 달라질 수 있습니다** **[추정: 코드의 순서 규칙에서 나온 해석]**.

In [2]:
def backtrace(ref, hyp, d):
    i, j = len(ref), len(hyp)
    ops = []
    while i > 0 or j > 0:
        if i > 0 and j > 0 and ref[i - 1] == hyp[j - 1] and d[i][j] == d[i - 1][j - 1]:
            ops.append(("일치", ref[i - 1], hyp[j - 1])); i -= 1; j -= 1
        elif i > 0 and j > 0 and d[i][j] == d[i - 1][j - 1] + 1:
            ops.append(("치환", ref[i - 1], hyp[j - 1])); i -= 1; j -= 1
        elif i > 0 and d[i][j] == d[i - 1][j] + 1:
            ops.append(("삭제", ref[i - 1], None)); i -= 1
        else:
            ops.append(("삽입", None, hyp[j - 1])); j -= 1
    return ops[::-1]

d = edit_matrix(ref, hyp)
ops = backtrace(ref, hyp, d)
pd.DataFrame(ops, columns=["종류", "정답 쪽", "인식 쪽"])

,종류,정답 쪽,인식 쪽
0,일치,the,the
1,일치,cat,cat
2,치환,sat,sit
3,일치,on,on
4,삭제,the,NaN
5,일치,mat,mat


위 표에서 `치환`이 1개(`sat` → `sit`), `삭제`가 1개(`the`)입니다. 이 개수를 세면 $S, D, I$ 가 나옵니다.

## 4. WER · CER 함수 만들기

- 입력은 **이미 토큰으로 나눈 리스트**로 받고, 문장 → 토큰은 따로 함수로 만듭니다(WER는 단어, CER는 글자).
- 정답이 비어 있으면 분모 $N=0$ 이라 계산할 수 없으므로 **오류로 처리**합니다.

In [3]:
def score(ref_tokens, hyp_tokens):
    if len(ref_tokens) == 0:
        raise ValueError("정답이 비어 있어 오류율을 계산할 수 없습니다 (분모 N = 0)")
    d = edit_matrix(ref_tokens, hyp_tokens)
    ops = backtrace(ref_tokens, hyp_tokens, d)
    S = sum(o[0] == "치환" for o in ops)
    D = sum(o[0] == "삭제" for o in ops)
    I = sum(o[0] == "삽입" for o in ops)
    N = len(ref_tokens)
    distance = d[-1][-1]
    assert S + D + I == distance          # 검산: 역추적한 오류 수의 합 = 편집 거리
    return {"S": S, "D": D, "I": I, "N": N, "거리": distance, "오류율": distance / N, "ops": ops}

def wer(ref, hyp):
    return score(ref.split(), hyp.split())

def cer(ref, hyp):
    # 이 노트북의 선택: 공백을 빼고 글자 단위로 비교 (공백을 글자로 세는 방식도 있음 [일반 지식])
    return score(list(ref.replace(" ", "")), list(hyp.replace(" ", "")))

ref_s = "the cat sat on the mat"
hyp_s = "the cat sit on mat"
for name, f in [("WER", wer), ("CER", cer)]:
    r = f(ref_s, hyp_s)
    print(f"{name}: S={r['S']} D={r['D']} I={r['I']} N={r['N']}  →  ({r['S']}+{r['D']}+{r['I']})/{r['N']} = {r['오류율']:.3f}")

WER: S=1 D=1 I=0 N=6  →  (1+1+0)/6 = 0.333
CER: S=1 D=3 I=0 N=17  →  (1+3+0)/17 = 0.235


**손계산과 맞추기** — 공백을 뺀 글자열은 정답 `thecatsatonthemat`(17자), 인식 `thecatsitonmat`(14자)입니다. `sat`→`sit`에서 글자 1개 치환, `the` 3글자 삭제이므로 $S=1,\ D=3,\ I=0$ → $(1+3)/17 \approx 0.235$ 가 되어야 하고, 위 출력과 같은지 확인합니다(같은 거리의 다른 경로가 있어도 **합은 같습니다**).

In [4]:
assert wer(ref_s, hyp_s)["거리"] == 2 and abs(wer(ref_s, hyp_s)["오류율"] - 2/6) < 1e-9
assert cer(ref_s, hyp_s)["거리"] == 4 and abs(cer(ref_s, hyp_s)["오류율"] - 4/17) < 1e-9
print("손계산과 일치: WER 2/6, CER 4/17")

손계산과 일치: WER 2/6, CER 4/17


## 5. WER는 1을 넘을 수 있다

분모는 **정답 단어 수**인데, 인식 쪽에 **끼어든 단어(삽입)** 는 분자에만 더해집니다. 그래서 삽입이 많으면 1을 넘습니다.
따라서 화면에 `1 − WER` 을 "일치율"로 보여 주면 **음수**가 나올 수 있으므로, 프로젝트에서는 **오류율과 S·D·I 개수**로 표시하기로 했습니다.

In [5]:
r = wer("hello", "hello hello hello")
print(r["S"], r["D"], r["I"], r["N"], "→ WER =", r["오류율"], "  (1 - WER =", 1 - r["오류율"], ")")

0 0 2 1 → WER = 2.0   (1 - WER = -1.0 )


## 6. 단어 단위 vs 글자 단위

| 언어 | 단어 단위(WER)의 문제 | 글자 단위(CER)의 장점 |
|---|---|---|
| 영어 | 단어가 하나만 달라도 단어 전체가 틀림 (`love` vs `loved`) | 글자 하나 차이만 반영 |
| 한국어 | **조사**가 붙어 `날씨가`와 `날씨`가 서로 다른 단어 → WER가 가혹함 | 글자 단위로 차이가 작게 반영 |
| 일본어 | **띄어쓰기가 없어** 단어로 나누려면 별도 도구가 필요함 | 글자 단위(CER)로 바로 계산 가능 |

In [6]:
pairs = [
    ("영어", "i love you", "i loved you"),
    ("한국어", "오늘 날씨가 좋아요", "오늘 날씨 좋아요"),
]
rows = []
for lang, a, b in pairs:
    w, c = wer(a, b), cer(a, b)
    rows.append([lang, a, b, f"{w['거리']}/{w['N']} = {w['오류율']:.3f}", f"{c['거리']}/{c['N']} = {c['오류율']:.3f}"])
pd.DataFrame(rows, columns=["언어", "정답", "인식", "WER", "CER"])

,언어,정답,인식,WER,CER
0,영어,i love you,i loved you,1/3 = 0.333,1/8 = 0.125
1,한국어,오늘 날씨가 좋아요,오늘 날씨 좋아요,1/3 = 0.333,1/8 = 0.125


- `i love you` vs `i loved you`: 단어 하나가 통째로 틀려 WER는 $1/3$ 이지만, 공백을 뺀 글자 8개 중 `d` 하나만 늘었으므로 CER는 $1/8$ 로 더 작습니다(두 문장 모두 같은 값이 되었다는 점에 주의: 단어 수 3개 중 1개 vs 글자 8개 중 1개).
- `날씨가` vs `날씨`: 조사 하나 때문에 WER $= 1/3$ 이지만 CER $= 1/8$ 입니다.
- 일본어는 단어 분할이 필요해서 프로젝트 계획에서도 **CER 중심**으로 두었습니다.

## 7. 정규화가 점수를 바꾼다

STT는 대소문자·구두점을 붙여서 내놓기도 합니다. 정답과 **글자가 같아도 형식이 다르면 오류로 센다**는 문제가 생기므로, 비교 전에 **같은 규칙으로 정리(정규화)** 합니다.

In [7]:
ref_raw = "Hello, World!"
hyp_raw = "hello world"

before = wer(ref_raw, hyp_raw)
print("정규화 전: WER =", before["오류율"], f"(S={before['S']} D={before['D']} I={before['I']})")

정규화 전: WER = 1.0 (S=2 D=0 I=0)


In [8]:
import re, unicodedata

def normalize_en(text):
    """영어 정규화 예시: NFC → 소문자 → 구두점 제거(아포스트로피는 유지) → 연속 공백 정리"""
    text = unicodedata.normalize("NFC", text).lower()
    text = re.sub(r"[^\w\s']", "", text)
    return re.sub(r"\s+", " ", text).strip()

after = wer(normalize_en(ref_raw), normalize_en(hyp_raw))
print("정규화 후: WER =", after["오류율"])
print(normalize_en("Don't  stop,   believin'!"))

정규화 후: WER = 0.0
don't stop believin'


- 같은 소리를 인식했는데 정규화 전에는 WER $=1.0$, 후에는 $0$ 입니다. **점수가 아니라 규칙이 결과를 바꾼 것**이므로, 프로젝트에서는 규칙을 고정하고 문서화하기로 했습니다.
- 위 규칙은 **예시**입니다. 축약형(`don't`)·숫자 표기를 어떻게 다룰지는 프로젝트 1일차에 결정합니다.
- 참고(검토 문서 7절에서 확인한 값): 수업 제공 한국어 WAV 5개를 Whisper tiny로 인식한 합산 결과는 **구두점을 제거하면 WER 0.060 / CER 0.034, 그대로 두면 WER 0.200** 이었습니다. 이 노트북에서는 다시 실행하지 않았습니다.

## 8. `difflib`로 센 오류는 편집 거리와 다를 수 있다

파이썬 표준 라이브러리 `difflib.SequenceMatcher`는 두 열의 **눈에 띄는 차이**를 보여 주는 도구이고, **최소 편집 경로를 보장하는 알고리즘은 아닙니다** **[일반 지식 + 아래 실험으로 확인]**. 그래서 `difflib` 결과로 오류 개수를 세면 WER 계산(편집 거리)과 **어긋날 수 있습니다.**

In [9]:
import difflib, random

def difflib_cost(a, b):
    sm = difflib.SequenceMatcher(None, a, b, autojunk=False)
    cost = 0
    for tag, i1, i2, j1, j2 in sm.get_opcodes():
        if tag == "equal":
            continue
        cost += max(i2 - i1, j2 - j1)       # 치환은 긴 쪽 길이, 삭제·삽입은 해당 길이
    return cost

a = ['b', 'c', 'b', 'a', 'c', 'a', 'a']
b = ['c', 'b', 'c', 'c', 'c', 'a']
print("difflib로 센 오류:", difflib_cost(a, b), " / 편집 거리:", edit_matrix(a, b)[-1][-1])

# 무작위 3글자 열 20000쌍으로 비교 (아주 반복적인 열이라 극단적인 경우)
random.seed(0)
mismatch = 0
for _ in range(20000):
    x = [random.choice("abc") for _ in range(random.randint(3, 8))]
    y = [random.choice("abc") for _ in range(random.randint(3, 8))]
    if difflib_cost(x, y) != edit_matrix(x, y)[-1][-1]:
        mismatch += 1
print("어긋난 쌍:", mismatch, "/ 20000")

difflib로 센 오류: 4  / 편집 거리: 3


어긋난 쌍: 8429 / 20000


- 위 실험은 **3글자만 쓰는 무작위 열**이라 극단적인 경우입니다. 실제 문장에서 얼마나 자주 어긋나는지는 **측정하지 않았습니다.**
- 그래도 "점수(편집 거리)와 화면의 오류 표시가 **같은 경로**여야 한다"는 이유로, 프로젝트에서는 **역추적으로 얻은 치환·삭제·삽입**을 표시에 쓰고 `difflib`은 보조로만 씁니다.

## 9. 연습 문제

**먼저 종이에 직접** $S,\ D,\ I,\ N$ 과 WER를 계산한 뒤, 아래 확인 셀의 결과와 비교해 보세요.

| 문제 | 정답(reference) | 인식(hypothesis) |
|---|---|---|
| ① | i want to explore the universe | i want explore a universe |
| ② | she sells sea shells | she sells sea shells by the shore |
| ③ | 오늘 날씨가 좋아요 | 오늘 날씨 좋아요 (CER도 계산) |

In [10]:
problems = [
    ("①", "i want to explore the universe", "i want explore a universe", "WER"),
    ("②", "she sells sea shells", "she sells sea shells by the shore", "WER"),
    ("③", "오늘 날씨가 좋아요", "오늘 날씨 좋아요", "CER"),
]
for no, a, b, kind in problems:
    r = (wer if kind == "WER" else cer)(a, b)
    print(f"{no} {kind}: S={r['S']} D={r['D']} I={r['I']} N={r['N']} → {r['거리']}/{r['N']} = {r['오류율']:.3f}")

① WER: S=1 D=1 I=0 N=6 → 2/6 = 0.333
② WER: S=0 D=0 I=3 N=4 → 3/4 = 0.750
③ CER: S=0 D=1 I=0 N=8 → 1/8 = 0.125


**풀이 확인**

- ① 정답 6단어. `to` 삭제 1, `the`→`a` 치환 1 → $(1+1)/6 = 0.333$
- ② 정답 4단어. 인식에 `by the shore` 3단어가 **삽입** → $3/4 = 0.75$ (삽입은 분자에만 더해짐)
- ③ 공백을 뺀 정답은 8글자, 조사 `가` 하나가 삭제 → $1/8 = 0.125$

(세 값이 위 출력과 다르면 어디서 갈렸는지 표와 역추적을 다시 따라가 봅니다.)

## 10. 정리와 프로젝트 연결

**핵심 정리**

- $\text{WER} = (S+D+I)/N$, $N$ 은 **정답**의 단어 수. CER는 글자 단위로 같은 계산.
- $S+D+I$ 는 **편집 거리**이고, 표(DP)로 구하고 **역추적**으로 어디가 틀렸는지 얻는다.
- 삽입이 많으면 **1을 넘을 수 있다** → "일치율 = 1 − WER"로 쓰지 않는다.
- 정답이 비면 계산 불가 → 오류로 처리.
- 정규화 규칙(대소문자·구두점·공백·유니코드 등)에 따라 점수가 크게 달라진다 → 규칙을 고정하고 기록.
- **주의**: WER·CER는 "인식 결과가 정답 문장과 얼마나 다른가"이지 **발음 점수가 아니다.** STT는 문맥으로 틀린 발음을 올바른 문장으로 고쳐 적을 수 있다.

**프로젝트 `scoring.py` 로 옮길 때의 목록**

| 함수 | 역할 |
|---|---|
| `normalize(text, lang)` | 언어별 정규화 |
| `edit_matrix`, `backtrace` | 편집 거리 표와 역추적 |
| `score(ref_tokens, hyp_tokens)` | S·D·I·N·오류율 반환, 빈 정답 처리 |
| `wer`, `cer` | 단어/글자 단위 호출 |
| 테스트 | 손계산 5건, 경계 사례(빈 정답, 동일 문장, WER > 1), `S+D+I == 거리` 검산 |